# 02 — Image Denoising with a Convolutional Autoencoder

**Goal:** Inject Gaussian noise into Fashion-MNIST images, then train a convolutional autoencoder to recover the clean originals. Measure improvement with **PSNR** (peak signal-to-noise ratio).

**Why convolutional:** dense autoencoders ignore spatial structure. Conv layers preserve neighborhood information, which is exactly what denoising needs.

**Author:** Zain Rafeeque

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)
print('TensorFlow', tf.__version__)

## 1. Load and corrupt Fashion-MNIST

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32')[..., None] / 255.0   # (N, 28, 28, 1)
x_test  = x_test.astype('float32')[..., None]  / 255.0

NOISE_STD = 0.4
x_train_noisy = np.clip(x_train + NOISE_STD * np.random.normal(size=x_train.shape), 0, 1).astype('float32')
x_test_noisy  = np.clip(x_test  + NOISE_STD * np.random.normal(size=x_test.shape),  0, 1).astype('float32')

print('shapes:', x_train.shape, '/ noisy', x_train_noisy.shape)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for i in range(6):
    axes[0, i].imshow(x_test[i, ..., 0], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_test_noisy[i, ..., 0], cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_title('clean', loc='left')
axes[1, 0].set_title(f'+N(0, {NOISE_STD})', loc='left')
plt.tight_layout(); plt.show()

## 2. Convolutional autoencoder

Encoder: two `Conv2D + MaxPool` blocks → 7×7×8 latent map.
Decoder: two `Conv2DTranspose` blocks back to 28×28×1.

In [ ]:
def build_denoiser():
    inp = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D(2, padding='same')(x)
    x = layers.Conv2D(8, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)              # latent: 7x7x8
    x = layers.Conv2DTranspose(8, 3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, activation='relu', padding='same')(x)
    out = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)
    return models.Model(inp, out, name='conv_denoiser')

denoiser = build_denoiser()
denoiser.compile(optimizer='adam', loss='mse')
denoiser.summary()

## 3. Train (input = noisy, target = clean)

In [ ]:
history = denoiser.fit(
    x_train_noisy, x_train,
    epochs=12,
    batch_size=256,
    validation_data=(x_test_noisy, x_test),
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=2,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Denoising MSE'); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. PSNR before vs after

Higher PSNR = closer to the clean reference. Each +6 dB roughly halves the perceived noise level.

In [ ]:
def psnr(clean, recon):
    mse = np.mean((clean - recon) ** 2, axis=(1, 2, 3))
    return 10 * np.log10(1.0 / np.maximum(mse, 1e-12))

denoised = denoiser.predict(x_test_noisy, batch_size=256, verbose=0)

psnr_noisy   = psnr(x_test, x_test_noisy).mean()
psnr_denoised = psnr(x_test, denoised).mean()
improvement  = psnr_denoised - psnr_noisy

print(f'PSNR (noisy   vs clean): {psnr_noisy:.2f} dB')
print(f'PSNR (denoised vs clean): {psnr_denoised:.2f} dB')
print(f'Improvement: +{improvement:.2f} dB')

## 5. Visual proof

In [ ]:
n = 8
idx = np.random.choice(len(x_test), n, replace=False)
fig, axes = plt.subplots(3, n, figsize=(n * 1.6, 5))
for i, k in enumerate(idx):
    axes[0, i].imshow(x_test[k, ..., 0], cmap='gray');       axes[0, i].axis('off')
    axes[1, i].imshow(x_test_noisy[k, ..., 0], cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(denoised[k, ..., 0], cmap='gray');     axes[2, i].axis('off')
axes[0, 0].set_title('clean',    loc='left')
axes[1, 0].set_title('noisy',    loc='left')
axes[2, 0].set_title('denoised', loc='left')
plt.tight_layout(); plt.show()

## Takeaways

- The convolutional bottleneck (7×7×8) preserves enough spatial information to recover edges that pure-dense bottlenecks would smear.
- We get a measurable PSNR improvement of several dB — the model isn't just memorizing.
- The same architecture generalizes to higher noise levels (try `NOISE_STD = 0.6`) at the cost of some lost fine detail.